In [ ]:
%%capture
!pip install advertools

In [ ]:
import pandas as pd
pd.options.display.max_columns = None
import re
import os
import time
from tqdm import tqdm

from dataset_utilities import value_counts_plus

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Sample lines from the log file

In [ ]:
!ls -lsSh /kaggle/input/web-server-access-logs/access.log

In [ ]:
!head -n 4 /kaggle/input/web-server-access-logs/access.log

Dataset
The dataset was downloaded from Harvard's Dataverse and contains logs from an Iranian ecommerce site (zanbil.ir):

Zaker, Farzin, 2019, "Online Shopping Store - Web Server Logs",  https://doi.org/10.7910/DVN/3QBYB5, Harvard Dataverse, V1

Log Format¶
This approach assumes the common log format and/or the combined one, which are two of the most commonly used. Eventually other formats can be incorporated. We start with the below regular express taken from:

Regular Expressions Cookbook
by Jan Goyvaerts, Steven Levithan
Publisher: O'Reilly Media, Inc. Release Date: August 2012

In [ ]:
# There is a minor bug in this regex, it misses the last field. I'll fix this soon. 

common_regex = '^(?P<client>\S+) \S+ (?P<userid>\S+) \[(?P<datetime>[^\]]+)\] "(?P<method>[A-Z]+) (?P<request>[^ "]+)? HTTP/[0-9.]+" (?P<status>[0-9]{3}) (?P<size>[0-9]+|-)'
combined_regex = '^(?P<client>\S+) \S+ (?P<userid>\S+) \[(?P<datetime>[^\]]+)\] "(?P<method>[A-Z]+) (?P<request>[^ "]+)? HTTP/[0-9.]+" (?P<status>[0-9]{3}) (?P<size>[0-9]+|-) "(?P<referrer>[^"]*)" "(?P<useragent>[^"]*)'
columns = ['client', 'userid', 'datetime', 'method', 'request', 'status', 'size', 'referer', 'user_agent']

The Approach
Loop through the lines of the input log file one by one. This ensures minimal memory consumption.
For each line, check it against the regular expression, and process it:
Match: append the matched line to a parsed_lines list
No match: append the non-matching line to the errors_file for later analysis
Once parsed_lines reaches 250,000 elements, convert the list to a DataFrame and save it to a parquet file in the output_dir. Clear the list. This also ensures minimal memory usage, and the 250k can be tweaked if necessary.
Read all the files of the output_dir with read_parquet into a pandas DataFrame. This function handles reading all the files and combines them.
Optimize the columns by using more efficient data types, most notably the pandas categorical type.
Write the DataFrame to a single file, for more convenient handling, and with the more efficient datatypes. This results in even faster reading.
Delete the files in output_dir.
Read in the final file with read_parquet.
Start analyzing.
Create a destinatoin directory where output files will be stored¶

In [ ]:
%mkdir parquet_dir

The logs_to_df function

In [ ]:
import time
import re
import pandas as pd


def logs_to_df(logfile, output_dir, errors_file):
    with open(logfile) as source_file:
        linenumber = 0
        parsed_lines = []
        for line in tqdm(source_file):
            try:
                log_line = re.findall(combined_regex, line)[0]
                parsed_lines.append(log_line)
            except Exception as e:
                with open(errors_file, 'at') as errfile:
                    print((line, str(e)), file=errfile)
                continue
            linenumber += 1
            if linenumber % 250_000 == 0:
                df = pd.DataFrame(parsed_lines, columns=columns)
                df.to_parquet(f'{output_dir}/file_{linenumber}.parquet')
                parsed_lines.clear()
        else:
            df = pd.DataFrame(parsed_lines, columns=columns)
            df.to_parquet(f'{output_dir}/file_{linenumber}.parquet')
            parsed_lines.clear()

Times will vary from system to system, and I will use the approximate values, so when you read this, you will likely see slightly different numbers.

In [ ]:
%time logs_to_df(logfile='/kaggle/input/web-server-access-logs/access.log', output_dir='parquet_dir/', errors_file='errors.txt')


The whole process just described took around 2.5 minutes.

Actually we are now ready to start analysis, as we have the parquet files that can be read. But we will optimize them even more.

Checking the number of resulting parsing errors:

In [ ]:
!wc errors.txt

In [ ]:
%time logs_df = pd.read_parquet('parquet_dir/')

Reading the whole directory takes about nine seconds. We now check the size of the resulting directory on disk:

In [ ]:
!du -sh parquet_dir/

257 ÷ 3,300 = 0.07.

The resulting file is 7% the size of the original.

Let's see how much memory it takes:

In [ ]:
logs_df.info(show_counts=True, verbose=True)

711 MB. We now remove the files in parquet_dir and optimize the datatypes and use more efficient ones.

In [ ]:
logs_df['client'] = logs_df['client'].astype('category')
del logs_df['userid']
logs_df['datetime'] = pd.to_datetime(logs_df['datetime'], format='%d/%b/%Y:%H:%M:%S %z')
logs_df['method'] = logs_df['method'].astype('category')
logs_df['status'] = logs_df['status'].astype('int16')
logs_df['size'] = logs_df['size'].astype('int32')
logs_df['referer'] = logs_df['referer'].astype('category')
logs_df['user_agent'] = logs_df['user_agent'].astype('category')

In [ ]:
logs_df.info(verbose=True, show_counts=True)

The file was reduced further from 711 to 342 MB. (342 ÷ 711 = 0.48 of the original size)

We now save it to a single file, and read again.